In [1]:
import torch
import torch.nn as nn
from torch.profiler import profile, record_function, ProfilerActivity

from models import CoreModel

In [32]:
model =  CoreModel(n_channels=16,
        input_size=54,
        hidden_size=8,
        num_layers=1,
        backbone_type='linpoly',
        batch_size=128,
        out_filtration=False)

In [22]:
def run(model, name,shape=(128, 54, 16, 2)):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)
    x = torch.randn(shape).to(device)
    model.to(device)
    
    # Initialize profiler with proper configuration
    with profile(
        activities=[ProfilerActivity.CUDA, ProfilerActivity.CPU]
        if torch.cuda.is_available() else [ProfilerActivity.CPU],
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=3),
        record_shapes=True,
        profile_memory=True,
        #on_trace_ready=torch.profiler.tensorboard_trace_handler('./logs'),
        with_stack=True
    ) as prof:
        
        for step in range(5):  # Run a few steps
            with record_function("forward"):
                output = model(x)
            prof.step()  # Properly handle profiler steps
    
    # Print profiling results sorted by CUDA time
    print(prof.key_averages().table(
        sort_by="cuda_time_total" if torch.cuda.is_available() else "cpu_time_total",
        row_limit=10
    ))

    print(prof.key_averages(group_by_input_shape=True).table(sort_by="cpu_time_total", row_limit=15))

    prof.export_chrome_trace(f"{name}.json")

In [23]:
run(model,name="leaklinear")

cuda
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us        1.422s      6056.00%        1.422s     473.889ms           0 b           0 b           0 b        

In [33]:
run(model, name='linpoly')

cuda
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us      15.641ms      1922.32%      15.641ms       5.214ms           0 b           0 b           0 b        